In [1]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

In [2]:
banners = pd.read_csv("../data/db/banners.csv", parse_dates=["created_at"])
interactions = pd.read_csv("../data/db/banner_interactions.csv", parse_dates=["event_date"])
users = pd.read_csv("../data/db/users.csv")

In [3]:
df = interactions.merge(users, on="user_id", how="left")
df = df.merge(banners, on="banner_id", how="left")

In [4]:
df["target_ctr"] = df["clicks"] / df["impressions"]

In [ ]:
# Возраст в целевом диапазоне
# Совпадение по возрасту
df["age_match"] = (
    (df["age"] >= df["target_age_min"]) &
    (df["age"] <= df["target_age_max"])
).astype(int)

In [ ]:
df["age_distance_to_target"] = 0
df.loc[df["age"] < df["target_age_min"], "age_distance_to_target"] = (
    df["target_age_min"] - df["age"]
)
df.loc[df["age"] > df["target_age_max"], "age_distance_to_target"] = (
    df["age"] - df["target_age_max"]
)

In [ ]:
df.head()

In [8]:
# Пол в целевом диапазоне
# Совпадение по полу
df["gender_match"] = (
    (df["target_gender"] == "U") |
    (df["gender"] == df["target_gender"]) |
    (df["gender"] == "U")
).astype(int)

In [10]:
df["gender_match"] = (
    (df["target_gender"] == "all") |
    (df["gender"] == df["target_gender"])
).astype(int)

In [11]:
df.head()

,event_date,user_id,banner_id,impressions,clicks,ctr,age,gender,city_tier,device_os,...,target_gender,target_age_min,target_age_max,cpm_bid,quality_score,created_at,is_active,landing_page,target_ctr,gender_match
0,2026-01-01,u_00007,b_0082,3,0,0.0,20,M,tier_2,Android,...,F,30,50,2.15,0.507,2025-12-30,1,https://example.com/food/snacks/orbit-82,0.0,0
1,2026-01-01,u_00009,b_0234,3,0,0.0,39,U,tier_2,iOS,...,U,25,35,5.08,0.949,2025-10-19,1,https://example.com/food/delivery/delta-234,0.0,1
2,2026-01-01,u_00012,b_0041,4,0,0.0,41,F,tier_1,Windows,...,M,40,60,10.95,0.843,2025-12-22,1,https://example.com/travel/hotels/aura-41,0.0,0
3,2026-01-01,u_00012,b_0215,5,1,0.2,41,F,tier_1,Windows,...,F,30,50,2.90,0.757,2025-12-24,1,https://example.com/entertainment/events/vanta...,0.2,1
4,2026-01-01,u_00015,b_0210,5,2,0.4,46,F,tier_3,Android,...,U,35,60,7.41,0.559,2025-10-28,1,https://example.com/finance/insurance/vanta-210,0.4,0


In [ ]:
df["interest_match"] = (
    (df["interest_1"] == df["subcategory"]) |
    (df["interest_2"] == df["subcategory"]) |
    (df["interest_3"] == df["subcategory"])
).astype(int)

In [ ]:
df["banner_age_days"] = (
    df["event_date"] - df["created_at"]
).dt.days.clip(lower=0)

df["event_dow"] = df["event_date"].dt.dayofweek

In [ ]:
train_df = df[df["event_date"] < "2026-03-16"].copy()
valid_df = df[df["event_date"] >= "2026-03-16"].copy()

In [ ]:
global_ctr = train_df["clicks"].sum() / train_df["impressions"].sum()

In [ ]:
features = [
    "user_id",
    "banner_id",
    "age",
    "gender",
    "city_tier",
    "device_os",
    "platform",
    "income_band",
    "activity_segment",
    "interest_1",
    "interest_2",
    "interest_3",
    "signup_days_ago",
    "is_premium",
    "brand",
    "category",
    "subcategory",
    "banner_format",
    "campaign_goal",
    "target_gender",
    "target_age_min",
    "target_age_max",
    "cpm_bid",
    "quality_score",
    "is_active",
    "age_match",
    "gender_match",
    # "interest_match",
    # "banner_age_days",
    # "event_dow",
    # "banner_ctr_prior",
    # "user_ctr_prior",
    # "subcategory_ctr_prior",
    # "user_subcat_ctr_prior",
]

cat_features = [
    "user_id",
    "banner_id",
    "gender",
    "city_tier",
    "device_os",
    "platform",
    "income_band",
    "activity_segment",
    "interest_1",
    "interest_2",
    "interest_3",
    "brand",
    "category",
    "subcategory",
    "banner_format",
    "campaign_goal",
    "target_gender",
]

In [ ]:
# 8. Обучение
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=200,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=10
)

In [ ]:
model.fit(
    train_df[features],
    train_df["target_ctr"],
    cat_features=cat_features,
    sample_weight=train_df["impressions"],
    eval_set=(valid_df[features], valid_df["target_ctr"]),
    use_best_model=True,
    early_stopping_rounds=10
)